# 11 — ag-ui Server

lionag2 serves over the **ag-ui protocol** — a standard SSE-based interface for agent UIs. AG2 has [native ag-ui support](https://docs.ag2.ai/latest/docs/beta/ag-ui/); lionag2 builds on `AGUIStream` directly.

Compatible with CopilotKit, Vercel AI SDK, and any ag-ui frontend.

```bash
lionag2-server --port 8000
# or
uvicorn lionag2.server:app
```

In [ ]:
import inspect
from lionag2.research.server import _build_coordinator, app

## The coordinator

`_build_coordinator()` assembles a coordinator agent with all specialists exposed as tools via `Agent.as_tool()`. Each specialist runs on a `persistent_stream()` — a shared stream that persists across tool calls.

```python
sf = persistent_stream()
for spec in roster:
    agent = Agent(spec["name"], ...)
    specialist_tools.append(
        agent.as_tool(
            description=f"{spec['role']}. Delegate: {spec['name']}.",
            stream=sf,
        )
    )

coordinator = Agent(
    "coordinator",
    prompt="Delegate to specialists in order.",
    tools=specialist_tools,
    response_schema=PromptedSchema(ExplorationResult),
)
```

The coordinator's response is a structured `ExplorationResult`. `PromptedSchema` is needed because AG2 uses `strict: True` with OpenAI's response_format, and models with optional fields (defaults) get rejected by strict validation.

AG2 also provides [`AgentSpec`](https://github.com/ag2ai/ag2/blob/main/autogen/beta/spec.py) for JSON-serializable agent specifications — useful for dynamic agent generation. Dynamic agents as a builtin tool are coming soon.

## AGUIStream

AG2's `AGUIStream` wraps any agent into an ASGI endpoint:

```python
stream = AGUIStream(coordinator)
endpoint = stream.build_asgi()
```

It maps AG2 events (tool calls, model responses, task lifecycle) to ag-ui SSE events that frontends understand.

## Starlette app

The ASGI app is minimal — two routes with CORS middleware:

```python
app = Starlette(
    routes=[
        Route("/", agui_endpoint, methods=["POST"]),
        Route("/health", health, methods=["GET"]),
    ],
    middleware=[Middleware(CORSMiddleware, allow_origins=["*"], ...)],
)
```

In [ ]:
for route in app.routes:
    print(f"  {route.path} [{', '.join(getattr(route, 'methods', ['*']))}]")

## Health endpoint

```bash
curl http://localhost:8000/health
```

Returns which optional services are configured:

```json
{"status": "ok", "khive": true, "exa": true, "daytona": false}
```

## CLI entry points

Two entry points in `pyproject.toml`:

```toml
[project.scripts]
lionag2 = "lionag2.cli:main"          # direct research run
lionag2-server = "lionag2.server:serve" # ag-ui server
```

```bash
# Direct research (outputs paper to stdout)
lionag2 "What are failure modes of chain-of-thought?" --max-depth 2

# Server mode
lionag2-server --port 8000 --model gpt-5.4-mini
```

## Up next

Tutorial 12 — the capstone — walks through a full `ResearchEngine.run()` end-to-end.